# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Binary Classification.**

We are predicting a categorical outcome with two states: whether a newly generated AI article is at high risk of experiencing a negative traffic trend (*down*) or not. We classify to decide if human intervention is needed.

## 2. Target or proxy

**Target:** `target_is_down` (Binary: 1 or 0).

This label is derived from an **observed outcome** in our analytics data. We create it by applying a simple rule to the existing dataset: if the observed `trend_direction` is `'down'`, the target is 1; otherwise, it is 0.

## 3. Success metric

**Precision (on the 'down' class).**

The business action based on this model is sending flagged content to a human editor. Human time is expensive. If the model predicts an article will trend down (1), we want to be highly confident it's true so we don't waste money fixing content that would have performed fine anyway.

## 4. The unit of analysis, as a real dataframe

One row = One published content piece (article/URL).

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Define our AI flag and our binary target
df['is_ai'] = df['provider_used'].notna() | df['model_used'].notna()
df['target_is_down'] = (df['trend_direction'] == 'down').astype(int)

# Filter for AI content and select features for the ML model
ai_df = df[df['is_ai'] == True].copy()
ml_df = ai_df[['content_id', 'model_used', 'content_type', 'main_intent',
               'word_count', 'content_age_days', 'target_is_down']]

# Display the unit of analysis
ml_df.head()

,content_id,model_used,content_type,main_intent,word_count,content_age_days,target_is_down
0,content_304f48230142,gemini-2.5-flash,keyword article,transactional,3221.0,187,1
1,content_a1fb4e703a9e,gemini-3-flash-preview,keyword article,informational,2481.0,445,1
2,content_9aa793d4d895,gemini-2.5-flash,keyword article,informational,3515.0,141,1
4,content_d99b7a2d90ca,gemini-3-flash-preview,keyword article,informational,2803.0,263,1
5,content_d4084a4bc775,gemini-2.5-flash,keyword article,transactional,3080.0,147,1


## 5. Why ML beats a fixed rule here

Content decay isn't driven by a single factor. The pattern is too messy for a fixed rule like "IF model = gpt-4o-mini AND word_count < 1000 THEN fail". A decline in traffic is driven by the complex, non-linear interaction between the specific AI model used, the search intent, the length, and the age of the content. ML can capture these multi-dimensional interactions to provide a directional probability that an if-statement cannot.